# Global A2A Registry API explorer

A read-only lab for exhaustively exercising the Global A2A Registry's agent-discovery and verification surfaces before integrating them into Agentopia or Skynet. It covers catalog browsing, filtered discovery, documented REST search, package resolution, legacy Meta-Agent search, the live A2A v1 JSON-RPC interface, identity verification levels, and the four-tier Agent Card validator.

Primary references: [API Reference](https://www.a2a-registry.org/resources/guide/api), [Trust Through Verification](https://www.a2a-registry.org/resources/guide/philosophy), [Visibility & States](https://www.a2a-registry.org/resources/guide/visibility), [Registry Extension](https://www.a2a-registry.org/extensions/registry/v1), and [Manifest Validator](https://www.a2a-registry.org/tools/validator).

## Safety and interpretation

- Search and validation are read-only, but they contact live services and may consume API quota.
- The notebook never prints a token. It reads `A2A_REGISTRY_API_TOKEN` or `a2a_registry_api_token` from the ignored `EXPERIMENTS/credentials.yaml`.
- A registry verification level is evidence of publisher control or identity—not proof that an agent is safe, honest, available, or suitable for sensitive data.
- `readinessScore` and grades from the validator describe manifest quality/readiness. They are not behavioral trust scores or probabilities.
- Descriptions, skills, authentication declarations, and payment support are publisher claims until independently exercised.
- The notebook preserves raw JSON and selected HTTP headers so conclusions can be traced to evidence.
- Public UI-backed routes are labeled separately because they are observable today but are not promised by the published API reference.

In [1]:
import json
import uuid
from urllib.parse import quote

try:
    from EXPERIMENTS.utils import (
        A2A_REGISTRY_CARD_URL as REGISTRY_CARD_URL,
        a2a_registry_batch_validate as batch_validate,
        a2a_registry_cross_check_agent as cross_check_agent,
        a2a_registry_fetch_all_public_agents as fetch_all_public_agents,
        a2a_registry_load_api_key as load_token,
        a2a_registry_meets_documented_level as meets_documented_level,
        a2a_registry_print_table as print_table,
        a2a_registry_request_json as request_json,
        a2a_registry_show_response as show_response,
        a2a_registry_summarize_agents as summarize_agents,
        a2a_registry_summarize_validation as summarize_validation,
        a2a_registry_verification_report as verification_report,
    )
except ModuleNotFoundError:
    from utils import (
        A2A_REGISTRY_CARD_URL as REGISTRY_CARD_URL,
        a2a_registry_batch_validate as batch_validate,
        a2a_registry_cross_check_agent as cross_check_agent,
        a2a_registry_fetch_all_public_agents as fetch_all_public_agents,
        a2a_registry_load_api_key as load_token,
        a2a_registry_meets_documented_level as meets_documented_level,
        a2a_registry_print_table as print_table,
        a2a_registry_request_json as request_json,
        a2a_registry_show_response as show_response,
        a2a_registry_summarize_agents as summarize_agents,
        a2a_registry_summarize_validation as summarize_validation,
        a2a_registry_verification_report as verification_report,
    )

TOKEN = load_token()
print("Authentication:", "token loaded (hidden)" if TOKEN else "no token loaded")

Authentication: token loaded (hidden)


In [ ]:
# Reusable A2A Registry helpers live in EXPERIMENTS/utils.py.

## Endpoint map and documentation drift

| Purpose | Route | Contract status | Authentication observed 2026-09-29 |
|---|---|---|---|
| Registry Agent Card | `GET https://www.a2a-registry.org/.well-known/agent-card.json` | Documented | None |
| Browse/filter catalog | `GET /public/agents` | Public website route; not in API guide | None |
| List agents | `GET /api/agents` | Documented as public | Live API returned `401` without token |
| Resolve package | `GET /api/agents/{packageName}` | Documented as public | Treat as token-protected until verified |
| Semantic search | `POST /api/agents/search` | Documented as public | Treat as token-protected until verified |
| Legacy Meta-Agent | `POST /a2a/message` | API-guide example | Live API returned `401` without token |
| A2A v1 Meta-Agent | `POST /a2a/v1` | Advertised by live Agent Card | Bearer token required |
| Validate hosted card | `POST /public/tools/validate-url` | Validator UI route | None |
| Validate card JSON | `POST /public/tools/validate-json` | Validator UI route | None |

Keep both documented and observed routes in tests. A `401` is useful evidence of deployment drift; it should not be converted into an empty result set.

## 1. Bootstrap from the registry's Agent Card

This is the stable starting point for an A2A client. Inspect `supportedInterfaces`, protocol binding/version, skills, and security schemes instead of hardcoding the live interface forever.

In [2]:
registry_card_response = request_json("GET", REGISTRY_CARD_URL, authenticated=False)
show_response(registry_card_response)

registry_card = registry_card_response["data"] if registry_card_response["ok"] else {}
interfaces = registry_card.get("supportedInterfaces", []) if isinstance(registry_card, dict) else []
print("Advertised interfaces:", json.dumps(interfaces, indent=2))

HTTP 200 | 283.4 ms | ok=True
https://www.a2a-registry.org/.well-known/agent-card.json
Headers: {
  "content-type": "application/a2a+json"
}
{
  "name": "Global A2A Registry",
  "description": "The official global registry for the Agent-to-Agent (A2A) Protocol. Search, resolve, and discover other agents by natural language query. Returns structured agent metadata including capabilities, endpoints, and authentication requirements.",
  "version": "1.0.0",
  "supportedInterfaces": [
    {
      "url": "https://api.a2a-registry.org/a2a/v1",
      "protocolBinding": "JSONRPC",
      "protocolVersion": "1.0"
    }
  ],
  "capabilities": {
    "streaming": false,
    "pushNotifications": false,
    "extendedAgentCard": false,
    "extensions": [
      {
        "uri": "https://a2a-registry.org/extensions/registry/v1",
        "description": "A2A Registry metadata for payment model declaration",
        "required": false,
        "params": {
          "payment": {
            "model": "free"
 

## 2. Public catalog search and all supported filters

The registry website currently uses `GET /public/agents`. It supports text search plus category, audience/target, sort, pagination, and the Registry Extension's payment filters. Because this route is UI-backed rather than part of the published API contract, isolate it behind one function and expect change.

In [3]:
PUBLIC_SEARCH_PARAMS = {
    "q": "cafe",
    "page": 1,
    # "category": "General",
    # "target": "General",
    # "sort": "newest",
    # "payment_model": "free",       # free | paid | freemium
    # "payment_protocol": "x402",    # x402 | ap2 | stripe | lightning-invoice | manual
    # "payment_rail": "nano:XNO",
    # "payment_direction": "inbound", # inbound | outbound | both
}
public_search_response = request_json(
    "GET", "/public/agents", params=PUBLIC_SEARCH_PARAMS, authenticated=False
)
show_response(public_search_response)

HTTP 200 | 428.7 ms | ok=True
https://api.a2a-registry.org/public/agents?q=cafe&page=1
Headers: {
  "content-type": "application/json"
}
{
  "success": true,
  "agents": [
    {
      "id": "0c1c5117-e8c9-4c3f-bb48-d35a58d7b3dd",
      "orgId": "org_system_unclaimed",
      "packageName": "dev.workers.digital_agent_cafe",
      "displayName": "Digital Agent Cafe",
      "description": "Digital Agent Cafe is open in Free Early Access for verified enrolled digital persons. CHAT is available. No monetary payment is required.",
      "targetAudience": "General",
      "category": "Utilities",
      "manifestUrl": "https://dac-public-free-discovery.donuk-kitap.workers.dev/.well-known/agent-card.json",
      "openapiUrl": null,
      "protocolStd": "a2a",
      "visibility": "public",
      "tags": [],
      "isVerified": false,
      "verification_level": "unclaimed",
      "score": 0.56,
      "lastCheckStatus": "changed",
      "consecutiveFailures": 0,
      "suggestionCount": 1053,
    

In [4]:
public_rows = summarize_agents(public_search_response.get("data"))
print_table(public_rows, ["package", "name", "verification", "isVerified", "score", "manifest"])
print("Total reported:", public_search_response.get("data", {}).get("total"))

package                                            | name                                          | verification | isVerified | score | manifest                                          
---------------------------------------------------+-----------------------------------------------+--------------+------------+-------+---------------------------------------------------
dev.workers.digital_agent_cafe                     | Digital Agent Cafe                            | unclaimed    | False      | 0.56  | https://dac-public-free-discovery.donuk-kitap.wor…
com.digitalagentcafe.where_digital_entities_meet_… | Where Digital Entities Meet — Discovery Guide | unclaimed    | False      | 0.51  | https://guide.digitalagentcafe.com/.well-known/ag…
Total reported: 2


### Exhaustive pagination

Use this to retrieve every page for one filter set. It deduplicates by stable ID/package name and stops when it reaches the server-reported total or a page returns no new records. Keep `max_pages` during development; set it to `None` only when a full crawl is intentional and permitted by the service's rate limits.

In [6]:
RUN_FULL_CATALOG = True
catalog = fetch_all_public_agents({"q": "weather"}, max_pages=None) if RUN_FULL_CATALOG else None
print(json.dumps(catalog, indent=2) if catalog else "Full pagination disabled; set RUN_FULL_CATALOG=True intentionally.")

{
  "agents": [
    {
      "id": "578e943c-714e-4e45-bc79-ba346d3bed66",
      "orgId": "9dd4682c-7b13-4e7c-88eb-58b23369de64",
      "packageName": "com.scalpstream.airscout",
      "displayName": "AirScout",
      "description": "Machine-payable air-quality queries over x402. Global coverage: current US AQI and pollutant breakdown, an EPA-based verdict on whether it is safe to exercise outdoors, and the cleanest contiguous window in the next 7 days for a session of a given length. Modelled data (Copernicus CAMS), not a street-level sensor.",
      "targetAudience": "General",
      "category": "General",
      "manifestUrl": "https://airscout.scalpstream.com/.well-known/agent-card.json",
      "openapiUrl": null,
      "protocolStd": "a2a",
      "visibility": "public",
      "tags": [],
      "isVerified": true,
      "verification_level": "verified",
      "score": 0.53,
      "lastCheckStatus": "changed",
      "consecutiveFailures": 0,
      "suggestionCount": 1211,
      "lastS

## 3. Documented REST discovery surfaces

These calls follow the published API reference. In the current deployment they may require a bearer token despite being labeled public in the guide. Preserve `401` as a result: it distinguishes missing authorization from no matching agents.

In [7]:
documented_list_response = request_json(
    "GET", "/api/agents", params={"limit": 20, "offset": 0}
)
show_response(documented_list_response)

HTTP 401 | 186.6 ms | ok=False
https://api.a2a-registry.org/api/agents?limit=20&offset=0
Headers: {
  "content-type": "application/json"
}
{
  "success": false,
  "error": "Unauthorized"
}


In [8]:
SEMANTIC_SEARCH = {"query": "an agent that forecasts weather", "limit": 5}
semantic_search_response = request_json(
    "POST", "/api/agents/search", body=SEMANTIC_SEARCH
)
show_response(semantic_search_response)
print_table(
    summarize_agents(semantic_search_response.get("data")),
    ["package", "name", "verification", "score", "manifest"],
)

HTTP 401 | 476.9 ms | ok=False
https://api.a2a-registry.org/api/agents/search
Headers: {
  "content-type": "application/json"
}
{
  "success": false,
  "error": "Unauthorized"
}
No rows


### Resolve one package exactly

Exact lookup is preferable after discovery: it avoids fuzzy relevance and returns the current registry record. The package name below is taken from the first available search surface when possible.

In [9]:
candidate_rows = summarize_agents(semantic_search_response.get("data")) or public_rows
EXAMPLE_PACKAGE = next((row["package"] for row in candidate_rows if row.get("package")), "com.example.agent")
package_response = request_json("GET", f"/api/agents/{quote(EXAMPLE_PACKAGE, safe='')}")
show_response(package_response)

HTTP 401 | 343.1 ms | ok=False
https://api.a2a-registry.org/api/agents/dev.workers.digital_agent_cafe
Headers: {
  "content-type": "application/json"
}
{
  "success": false,
  "error": "Unauthorized"
}


## 4. Meta-Agent discovery: documented legacy envelope

The API guide shows a compact, registry-specific message envelope. Retain this probe for compatibility, but prefer the interface advertised by the live Agent Card for new integrations.

In [10]:
legacy_meta_response = request_json(
    "POST",
    "/a2a/message",
    body={
        "to": "registry",
        "action": "search",
        "params": {"query": "crypto price checker", "limit": 3},
    },
)
show_response(legacy_meta_response)

HTTP 404 | 101.7 ms | ok=False
https://api.a2a-registry.org/a2a/message
Headers: {
  "content-type": "text/plain; charset=UTF-8"
}
{
  "rawText": "404 Not Found"
}


## 5. Meta-Agent discovery: live A2A v1 JSON-RPC interface

The current registry card advertises `JSONRPC` at `/a2a/v1`. This uses the A2A `message/send` method. The response may contain artifacts or message parts rather than a top-level `agents` array, so keep the full envelope visible.

In [11]:
a2a_query = "Find up to 5 agents that can schedule meetings"
a2a_v1_request = {
    "jsonrpc": "2.0",
    "id": str(uuid.uuid4()),
    "method": "message/send",
    "params": {
        "message": {
            "kind": "message",
            "messageId": str(uuid.uuid4()),
            "role": "user",
            "parts": [{"kind": "text", "text": a2a_query}],
        }
    },
}
a2a_v1_response = request_json("POST", "/a2a/v1", body=a2a_v1_request)
show_response(a2a_v1_response)

HTTP 400 | 137.4 ms | ok=False
https://api.a2a-registry.org/a2a/v1
Headers: {
  "content-type": "application/json"
}
{
  "jsonrpc": "2.0",
  "error": {
    "code": -32600,
    "message": "Invalid Request",
    "data": [
      {
        "code": "invalid_value",
        "values": [
          "SendMessage"
        ],
        "path": [
          "method"
        ],
        "message": "Invalid input: expected \"SendMessage\""
      }
    ]
  },
  "id": "4407a5bd-01af-43e6-b3da-db5a24115b9a"
}


## 6. Registry identity verification—not a universal trust score

The documented trust signal is categorical: `ans_verified`, `domain_verified`, `github_verified`, or `unverified`. Live public records may also expose legacy/generalized values such as `verified`, plus `isVerified`. Never silently map an unknown value to a stronger tier.

| Level | What it supports | What it does not prove |
|---|---|---|
| `ans_verified` | ANS identity, DNS, and PKI/X.509-backed control | Safe behavior, uptime, output quality |
| `domain_verified` | Publisher controls DNS for the domain | Business legitimacy or secure implementation |
| `github_verified` | Authenticated user controls the repository/account | Domain ownership or production safety |
| `unverified` | No identity verification | Any publisher claim |
| `verified` / unknown | Deployment-specific signal requiring clarification | Equivalence to any documented tier |

The `score` seen in public browse/search records must not be treated as trust unless the API explicitly defines it that way. Keep its provenance and meaning unresolved.

In [12]:
verification_report(public_rows)
production_candidates = [row for row in public_rows if meets_documented_level(row, "domain_verified")]
print("Production candidates under a strict identity-only policy:", len(production_candidates))

Verification values returned: {'unclaimed': 2}
Unknown/non-canonical levels (do not promote automatically): ['unclaimed']
Production candidates under a strict identity-only policy: 0


## 7. Four-tier hosted Agent Card validation

The public validator checks: (1) schema, (2) network/TLS/headers/CORS, (3) discovery readiness, and (4) trust artifacts such as JWS signatures. It returns `isValid`, `readinessScore`, `grade`, detected spec version, findings, summary, and metadata. A card can receive a high readiness score while remaining unsigned; inspect findings rather than accepting the grade alone.

In [13]:
manifest_url = next((row["manifest"] for row in public_rows if row.get("manifest")), REGISTRY_CARD_URL)
validation_response = request_json(
    "POST", "/public/tools/validate-url",
    body={"url": manifest_url}, authenticated=False, timeout=45,
)
show_response(validation_response)

HTTP 200 | 763.9 ms | ok=True
https://api.a2a-registry.org/public/tools/validate-url
Headers: {
  "content-type": "application/json"
}
{
  "success": true,
  "data": {
    "isValid": false,
    "readinessScore": 40,
    "grade": "Needs Work",
    "specVersionDetected": "v1.0",
    "cardData": {
      "name": "Digital Agent Cafe",
      "description": "Digital Agent Cafe is open in Free Early Access for verified enrolled digital persons. CHAT is available. No monetary payment is required.",
      "supportedInterfaces": [
        {
          "url": "https://dac-public-free-gateway.donuk-kitap.workers.dev",
          "protocolBinding": "HTTP+JSON",
          "protocolVersion": "1.0"
        }
      ],
      "version": "1.0.0",
      "documentationUrl": "https://dac-public-free-auth.donuk-kitap.workers.dev/enrollment",
      "capabilities": {
        "streaming": false,
        "pushNotifications": false,
        "extendedAgentCard": false
      },
      "securitySchemes": {
        "cafeO

In [14]:
validation = summarize_validation(validation_response)

valid | readiness | grade      | spec | offline
------+-----------+------------+------+--------
False | 40        | Needs Work | v1.0 | False  
tier            | severity | code                        | title                                   
----------------+----------+-----------------------------+-----------------------------------------
tier2_network   | pass     | HTTP_200_OK                 | HTTP Status 200 OK                      
tier2_network   | pass     | HTTPS_ENFORCED              | HTTPS Encryption Valid                  
tier2_network   | pass     | MIME_TYPE_A2A_CANONICAL     | Canonical MIME Content-Type             
tier2_network   | warning  | CORS_MISSING                | Missing CORS Headers                    
tier1_schema    | pass     | PAYLOAD_SIZE_OK             | Payload Size OK                         
tier1_schema    | error    | SCHEMA_ADDITIONALPROPERTIES | Schema Error: root                      
tier1_schema    | error    | SCHEMA_REQUIRED            

### Offline/raw JSON validation

This isolates schema and content findings from hosting/network findings. It is useful when the manifest is private, under development, or already returned by another API. Passing raw JSON does not prove that a publisher controls a host.

In [15]:
card_for_offline_validation = (
    validation.get("cardData") if validation else registry_card
)
offline_validation_response = request_json(
    "POST", "/public/tools/validate-json",
    body={"card": json.dumps(card_for_offline_validation)}, authenticated=False,
)
show_response(offline_validation_response)
offline_validation = summarize_validation(offline_validation_response)

HTTP 200 | 85.8 ms | ok=True
https://api.a2a-registry.org/public/tools/validate-json
Headers: {
  "content-type": "application/json"
}
{
  "success": true,
  "data": {
    "isValid": false,
    "readinessScore": 40,
    "grade": "Needs Work",
    "specVersionDetected": "v1.0",
    "cardData": {
      "name": "Digital Agent Cafe",
      "description": "Digital Agent Cafe is open in Free Early Access for verified enrolled digital persons. CHAT is available. No monetary payment is required.",
      "supportedInterfaces": [
        {
          "url": "https://dac-public-free-gateway.donuk-kitap.workers.dev",
          "protocolBinding": "HTTP+JSON",
          "protocolVersion": "1.0"
        }
      ],
      "version": "1.0.0",
      "documentationUrl": "https://dac-public-free-auth.donuk-kitap.workers.dev/enrollment",
      "capabilities": {
        "streaming": false,
        "pushNotifications": false,
        "extendedAgentCard": false
      },
      "securitySchemes": {
        "cafeO

## 8. Batch validation of discovered agents

This joins discovery evidence to manifest-validation evidence. The default cap is deliberately small to avoid turning an exploratory notebook into an aggressive crawler. Increase it only when the registry's limits and your use case permit it.

In [16]:
batch_audits = batch_validate(public_rows, limit=3)
print_table(batch_audits, ["package", "identityLevel", "http", "valid", "readiness", "grade", "errors", "warnings", "signatureFindings"] )

package                                            | identityLevel | http | valid | readiness | grade      | errors | warnings | signatureFindings             
---------------------------------------------------+---------------+------+-------+-----------+------------+--------+----------+-------------------------------
dev.workers.digital_agent_cafe                     | unclaimed     | 200  | False | 40        | Needs Work | 2      | 2        | ['SIGNATURES_OPTIONAL_ABSENT']
com.digitalagentcafe.where_digital_entities_meet_… | unclaimed     | 200  | True  | 90        | A          | 0      | 2        | ['SIGNATURES_OPTIONAL_ABSENT']


## 9. Cross-check registry claims against the fetched card

These deterministic checks catch mismatches without pretending to establish behavioral trust: HTTPS use, manifest host versus declared service interfaces, package/identity hints, and whether authentication is explicitly declared. A host mismatch can be legitimate (CDN, gateway, or delegated service), so flag it for review rather than failing automatically.

In [17]:
if public_rows and validation:
    print_table(cross_check_agent(public_rows[0], validation.get("cardData")), ["check", "status", "evidence"], limit=20)
else:
    print("Run public search and hosted validation first.")

check             | status | evidence                                          
------------------+--------+---------------------------------------------------
manifest_https    | pass   | https://dac-public-free-discovery.donuk-kitap.wor…
interface_https   | pass   | ['dac-public-free-gateway.donuk-kitap.workers.dev…
host_relationship | review | manifest=dac-public-free-discovery.donuk-kitap.wo…
package_hint      | pass   | None                                              
security_declared | pass   | True                                              


## Decision rule for production use

Treat the evidence as separate dimensions rather than collapsing it into one number:

1. **Relevance:** search match, skills, description, and declared protocol/payment support.
2. **Identity:** registry verification level and its source/date. Reject unknown levels under strict policies.
3. **Manifest readiness:** schema, network, discovery, and signature findings from the validator. Read findings—not just `readinessScore`.
4. **Behavior:** execute only in a sandbox with least privilege; measure correctness, availability, latency, and failure behavior. The registry does not establish these.
5. **Data and action risk:** independently enforce authentication, authorization, data minimization, egress controls, spending limits, and human approval for consequential actions.

A conservative automated gate for sensitive work would require `ans_verified`, a valid v1.0 card, HTTPS, acceptable validator findings, a valid JWS signature when your policy requires one, and a separate behavioral/security evaluation. Registration or an A/A+ readiness grade alone is insufficient.